In [2]:
"""
Doomscroll Detector
-------------------
Watches your webcam, spots mobile phones with a pretrained YOLO model,
and blasts a siren + funny warning if a phone stays visible too long.

Press 'q' or ESC in the video window to quit.
"""

import subprocess
import sys
import time

import cv2
import numpy as np
import pygame
from ultralytics import YOLO

# =====================================================================
# SETTINGS - change these to tweak the behaviour
# =====================================================================
CAMERA_INDEX = 0              # 0 = default webcam. Try 1 if you have several.
MODEL_NAME = "yolov8n.pt"     # Small, fast pretrained model (auto-downloads)
CONFIDENCE = 0.40             # 0-1. Lower = more detections, more false alarms
PHONE_CLASS_ID = 67           # "cell phone" in the COCO dataset

SECONDS_BEFORE_ALERT = 3.0    # Phone must be visible this long to trigger
GRACE_PERIOD = 0.7            # Phone can vanish this long without resetting timer
COOLDOWN_SECONDS = 20.0       # Minimum time between two alerts
ALERT_DISPLAY_SECONDS = 4.0   # How long the red warning flashes on screen

PLAY_DETECT_BEEP = True       # Short "bip-bip!" the moment a phone is first spotted
BEEP_GAP_SECONDS = 5.0        # Don't beep again more often than this
CUSTOM_ALERT_SOUND = None     # Or a file path, e.g. "my_sound.wav" or "bruh.mp3"
SPEAK_WARNING = True          # Use Windows built-in voice to read the warning
WARNING_TEXT = "STOP SCROLLING, GET BACK TO WORK!"
WINDOW_NAME = "Doomscroll Detector  (press Q to quit)"


# =====================================================================
# SOUND
# =====================================================================
def make_siren_sound(duration=3.0, sample_rate=44100):
    """Generate a loud, annoying police-style siren (no audio file needed)."""
    t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

    # Frequency sweeps between 600 Hz and 1400 Hz, 2 times per second
    freq = 1000 + 400 * np.sin(2 * np.pi * 2 * t)
    phase = 2 * np.pi * np.cumsum(freq) / sample_rate

    # Square-ish wave = harsher and louder than a pure sine wave
    wave = np.sign(np.sin(phase)) * 0.6 + np.sin(phase) * 0.4

    samples = (wave * 32767 * 0.9).astype(np.int16)
    stereo = np.ascontiguousarray(np.column_stack((samples, samples)))
    return pygame.sndarray.make_sound(stereo)


def make_beep_sound(sample_rate=44100):
    """A short double 'bip-bip!' that says 'I see your phone...'"""
    def tone(freq, length):
        t = np.linspace(0, length, int(sample_rate * length), endpoint=False)
        fade = np.minimum(1, np.minimum(t, length - t) * 100)  # avoids clicks
        return np.sin(2 * np.pi * freq * t) * fade

    gap = np.zeros(int(sample_rate * 0.08))
    wave = np.concatenate((tone(1000, 0.12), gap, tone(1400, 0.15)))
    samples = (wave * 32767 * 0.8).astype(np.int16)
    stereo = np.ascontiguousarray(np.column_stack((samples, samples)))
    return pygame.sndarray.make_sound(stereo)


def load_alert_sound():
    """Use your own sound file if set, otherwise the built-in siren."""
    if CUSTOM_ALERT_SOUND:
        try:
            sound = pygame.mixer.Sound(CUSTOM_ALERT_SOUND)
            sound.set_volume(1.0)
            return sound
        except Exception as error:
            print(f"Could not load '{CUSTOM_ALERT_SOUND}': {error}")
            print("Using the built-in siren instead.")
    return make_siren_sound()


def speak(text):
    """Read text aloud using Windows' built-in speech (no extra install)."""
    if not SPEAK_WARNING or sys.platform != "win32":
        return
    command = (
        "Add-Type -AssemblyName System.Speech; "
        "$s = New-Object System.Speech.Synthesis.SpeechSynthesizer; "
        "$s.Volume = 100; "
        f"$s.Speak('{text}')"
    )
    # Popen = run in background so the video doesn't freeze
    subprocess.Popen(
        ["powershell", "-NoProfile", "-Command", command],
        creationflags=subprocess.CREATE_NO_WINDOW,
    )


# =====================================================================
# DRAWING HELPERS
# =====================================================================
def draw_phone_boxes(frame, boxes):
    """Draw a green rectangle and label around each detected phone."""
    for (x1, y1, x2, y2, conf) in boxes:
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
        label = f"PHONE {conf:.0%}"
        cv2.putText(frame, label, (x1, max(y1 - 10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)


def draw_progress_bar(frame, progress):
    """Bar at the bottom showing how close you are to being roasted."""
    h, w = frame.shape[:2]
    bar_w = int((w - 40) * min(progress, 1.0))
    cv2.rectangle(frame, (20, h - 40), (w - 20, h - 20), (60, 60, 60), -1)
    colour = (0, 255 - int(255 * progress), int(255 * progress))  # green -> red
    cv2.rectangle(frame, (20, h - 40), (20 + bar_w, h - 20), colour, -1)
    cv2.putText(frame, "Doomscroll meter", (20, h - 48),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)


def draw_alert_overlay(frame):
    """Flashing red tint and big warning text."""
    h, w = frame.shape[:2]
    flash_on = int(time.time() * 4) % 2 == 0  # toggles 4 times per second
    if flash_on:
        red = np.zeros_like(frame)
        red[:] = (0, 0, 255)
        cv2.addWeighted(red, 0.35, frame, 0.65, 0, frame)

    for text, scale, y in [("STOP SCROLLING!", 1.8, h // 2 - 20),
                           ("GET BACK TO WORK!", 1.4, h // 2 + 50)]:
        size = cv2.getTextSize(text, cv2.FONT_HERSHEY_DUPLEX, scale, 4)[0]
        x = (w - size[0]) // 2
        cv2.putText(frame, text, (x, y), cv2.FONT_HERSHEY_DUPLEX,
                    scale, (0, 0, 0), 10)          # black outline
        cv2.putText(frame, text, (x, y), cv2.FONT_HERSHEY_DUPLEX,
                    scale, (255, 255, 255), 4)     # white text


# =====================================================================
# MAIN PROGRAM
# =====================================================================
def main():
    # --- Start sound system ---
    pygame.mixer.init(frequency=44100, size=-16, channels=2)
    siren = load_alert_sound()
    beep = make_beep_sound()

    # --- Load the pretrained YOLO model (downloads on first run) ---
    print("Loading YOLO model... (first run downloads ~6 MB)")
    model = YOLO(MODEL_NAME)

    # --- Open the webcam (CAP_DSHOW makes it start faster on Windows) ---
    cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)
    if not cap.isOpened():
        print("ERROR: Could not open the webcam. Is another app using it?")
        print("Tip: try changing CAMERA_INDEX to 1 at the top of this file.")
        return

    # Timers (all in seconds, from time.time())
    phone_first_seen = None   # when the current "phone streak" started
    phone_last_seen = 0.0     # last moment a phone was in view
    last_alert_time = 0.0     # when we last yelled at you
    alert_until = 0.0         # red overlay stays on until this time
    last_beep_time = 0.0      # when the "phone spotted" beep last played

    print("Running! Press 'q' or ESC in the video window to quit.")

    while True:
        ok, frame = cap.read()
        if not ok:
            print("Lost the webcam feed.")
            break

        now = time.time()

        # --- 1. Run YOLO, only looking for phones ---
        results = model(frame, conf=CONFIDENCE, classes=[PHONE_CLASS_ID],
                        verbose=False)[0]

        boxes = []
        for box in results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            boxes.append((x1, y1, x2, y2, float(box.conf[0])))

        # --- 2. Update the "how long has a phone been visible" timer ---
        if boxes:
            phone_last_seen = now
            if phone_first_seen is None:
                phone_first_seen = now
                # A phone just appeared: play a short beep as a heads-up
                # (skipped during cooldown or if we beeped very recently)
                alert_ready = (now - last_alert_time) >= COOLDOWN_SECONDS
                beep_ok = (now - last_beep_time) >= BEEP_GAP_SECONDS
                if PLAY_DETECT_BEEP and alert_ready and beep_ok:
                    beep.play()
                    last_beep_time = now
        elif phone_first_seen is not None and now - phone_last_seen > GRACE_PERIOD:
            phone_first_seen = None  # phone gone for good, reset

        visible_for = (now - phone_first_seen) if phone_first_seen else 0.0

        # --- 3. Trigger the alert (if the cooldown allows it) ---
        cooldown_done = (now - last_alert_time) >= COOLDOWN_SECONDS
        if visible_for >= SECONDS_BEFORE_ALERT and cooldown_done:
            print(f"ALERT! {WARNING_TEXT}")
            siren.play()
            speak(WARNING_TEXT)
            last_alert_time = now
            alert_until = now + ALERT_DISPLAY_SECONDS
            phone_first_seen = None  # restart the timer after the alert

        # --- 4. Draw everything ---
        draw_phone_boxes(frame, boxes)
        draw_progress_bar(frame, visible_for / SECONDS_BEFORE_ALERT)

        remaining = COOLDOWN_SECONDS - (now - last_alert_time)
        if last_alert_time and remaining > 0:
            status = f"Cooldown: {remaining:.0f}s"
        else:
            status = "Armed - hide the phone!" if boxes else "Watching..."
        cv2.putText(frame, status, (20, 30), cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, (255, 255, 0), 2)

        if now < alert_until:
            draw_alert_overlay(frame)

        cv2.imshow(WINDOW_NAME, frame)

        # --- 5. Quit on 'q' or ESC ---
        key = cv2.waitKey(1) & 0xFF
        if key in (ord("q"), ord("Q"), 27):
            break

    # --- Clean up ---
    cap.release()
    cv2.destroyAllWindows()
    pygame.mixer.quit()
    print("Bye! Now go be productive.")


if __name__ == "__main__":
    main()

Loading YOLO model... (first run downloads ~6 MB)
Running! Press 'q' or ESC in the video window to quit.
ALERT! STOP SCROLLING, GET BACK TO WORK!
ALERT! STOP SCROLLING, GET BACK TO WORK!
Bye! Now go be productive.
